In [1]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.backends.cudnn as cudnn  # noqa: PLR0402
import torch.nn as nn  # noqa: PLR0402
import torch.nn.functional as F
import torch.optim as optim  # noqa: PLR0402
import torchvision
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from torchvision import models, transforms

cudnn.benchmark = True
plt.ion()   # interactive mode
# PyTorch idiom for using GPU if available, otherwise fallback to CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Variables for training

In [21]:
SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = min(4, os.cpu_count() or 0)
MANIFEST_PATH = '../data/oxford-iiit-pet/manifest.json'



## Transforms and the data loading & Splitting 

In [3]:
# Standard ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(), # Simple data augmentation
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
}

In [19]:
import json

from PIL import Image
from torch.utils.data import Dataset


class PetManifestDataset(Dataset):
    def __init__(self, manifest_path, split=None, corruption=None, transform=None):
        """
        split: 'train' | 'test' | None (no filter)
        corruption: None -> only clean images (corruption is None)
                    'any' -> only corrupted images
                    'all' -> clean + corrupted
                    'gaussian_noise' etc -> only that corruption type
        """
        with open(manifest_path) as f:
            entries = json.load(f)

        if split is not None:
            entries = [e for e in entries if e["split"] == split]

        if corruption is None:
            entries = [e for e in entries if e["corruption"] is None]
        elif corruption == "any":
            entries = [e for e in entries if e["corruption"] is not None]
        elif corruption != "all":
            entries = [e for e in entries if e["corruption"] == corruption]
        # corruption == "all" -> no filter, keep everything

        # class_index can be None for some corrupted entries if lookup failed; drop those
        entries = [e for e in entries if e["class_index"] is not None]

        self.entries = entries
        self.transform = transform

        idx_to_name = {}
        for e in entries:
            idx_to_name[e["class_index"]] = e["breed"]
        self.classes = [idx_to_name[i] for i in sorted(idx_to_name)]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        img = Image.open(e["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, e["class_index"]

In [22]:
# Download and load the datasets

# from pet_manifest_dataset import PetManifestDataset

train_dataset = PetManifestDataset(
    manifest_path=MANIFEST_PATH,
    split='train',
    corruption=None,          # clean images only
    transform=data_transforms['train'],
)

test_dataset = PetManifestDataset(
    manifest_path=MANIFEST_PATH,
    split='test',
    corruption=None,
    transform=data_transforms['test'],
)

class_names = test_dataset.classes
print(len(class_names))

37


In [23]:
import random


# Set worker count dynamically based on available CPU cores (capped for safety)
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# 2. Tie generator seed to global SEED
g = torch.Generator()
g.manual_seed(SEED)

dataloaders = {
    'train': DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        worker_init_fn=seed_worker,
        generator=g,
    ),
    'val': DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
}

class_names = test_dataset.classes
print(len(class_names))

37


In [24]:
def imshow(inp, title=None):
    """Imshow for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)

# Get a batch of training data
inputs, classes = next(iter(dataloaders['train']))

# Make a grid from batch and display
out = torchvision.utils.make_grid(inputs[:5]) # Just show 5 images
imshow(out, title=[class_names[x] for x in classes[:5]])

FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/mennasalah/pet-breed-mlops-project/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 375, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/mennasalah/pet-breed-mlops-project/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_71351/672888737.py", line 46, in __getitem__
    img = Image.open(e["path"]).convert("RGB")
          ^^^^^^^^^^^^^^^^^^^^^
  File "/home/mennasalah/pet-breed-mlops-project/.venv/lib/python3.12/site-packages/PIL/Image.py", line 3639, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'da

In [10]:
import copy

from tqdm import tqdm


In [11]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=5):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in tqdm(dataloaders[phase], desc=f"{phase} epoch {epoch+1}"):
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            phase_size = len(dataloaders[phase].dataset)
            epoch_loss = running_loss / phase_size
            epoch_acc = running_corrects.double() / phase_size

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

In [12]:
# Load a pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False
    
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))

# Define the Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
# Observe that all parameters are being optimized
num_epocs = 5
# Decay LR by a factor of 0.1 every 2 epochs
scheduler = lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

model_ft = train_model(model, criterion, optimizer, scheduler,num_epocs)

Epoch 1/5
----------


train epoch 1:   0%|          | 0/115 [00:00<?, ?it/s]


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/mennasalah/pet-breed-mlops-project/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 375, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/mennasalah/pet-breed-mlops-project/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_71351/672888737.py", line 46, in __getitem__
    img = Image.open(e["path"]).convert("RGB")
          ^^^^^^^^^^^^^^^^^^^^^
  File "/home/mennasalah/pet-breed-mlops-project/.venv/lib/python3.12/site-packages/PIL/Image.py", line 3639, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'da

In [ ]:
def predict_top3(model, inputs, class_names):
    model.eval()  # inference mode
    with torch.no_grad():
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1)
        top3_probs, top3_idx = torch.topk(probs, 3, dim=1)  # top 3 per image

    results = []
    for i in range(inputs.size(0)):
        preds = [
            (class_names[top3_idx[i][j].item()], top3_probs[i][j].item())
            for j in range(3)
        ]
        results.append(preds)
    return results

In [ ]:
inputs, labels = next(iter(dataloaders['test']))
results = predict_top3(model_ft, inputs, class_names)

for i, preds in enumerate(results[:10]):  # first 5 images
    print(f"Image {i}:")
    print("    Top 3 predictions with confidence percentage:")
    for name, prob in preds:
        print(f"        {name}: {prob:.2%}")
    print(f"    TRUE: {class_names[labels[i]]}")


In [47]:
model_ft.eval()
running_loss = 0.0
running_corrects = 0

with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['test'], desc="test"):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model_ft(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

test_size = len(dataloaders['test'].dataset)
test_loss = running_loss / test_size
test_acc = running_corrects.double() / test_size

print(f'Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}')

test: 100%|██████████| 35/35 [00:49<00:00,  1.42s/it]

Test Loss: 0.4819 Acc: 0.8802
